First of all, since we will need some mathematical an plotting habilities not included in the standard Pyhton language, we have to import some modules. Execute the following cell

In [1]:
import numpy as np 
import matplotlib.pyplot as plt
from scipy.special import hermite
from scipy.special import genlaguerre


# Structured laser beams

The control of light in its transverse dimension has boosted the interest in the development of new techniques for structuring, sculpting, or tailoring light beams. The general term structured light refers to **light beams with non-homogeneous, non-trivial, distribution of intensity, phase and/or polarization properties in the plane transverse to propagation**.

The general expression for a beam propagating in along the $z$-axis contains a plane wave (_carrier wave_) whose amplitude and phase is modulated by an _envelope_, and a polarization vector:
\begin{equation}
\mathbf{E}(\mathbf{r}, t)= U(\mathbf{r}) e^{i (kz - \omega t)}  \mathbf{e}(\mathbf{r}, t)
\end{equation}

In this notebook we will develop, step by step, a basic program to represent structured laser beams, in the transverse plane $xy$. For simplicity, we shall consider the polarization vector constant, therefore irrelevant. We shall develop, therefore, a description for the scalar field. Also note that the _carrier wave_ is also a simple function, so the relevant information on the structure of the field is contained in the _envelope_. In our code, therefore, we shall focus, therefore, only in the description of the complex envelope function $U(\mathbf{r})$.


## A class to represent a spatial volume

Python is a highly structure language. Programming a structured code is the best way to facilitate code reuse, i.e. to build a complex code incrementally, from more simnple parts. The main architecture in Python is the _class_.

A class is a container that includes "local variables" (called _attributes_) and "local functions" (_methods_) restricted to the object environment.

So, just to begin, let us cosider a class named *spatial_volume*. This will contain the specifications for the spatial box where the field is defined, and that we will use to represent spatially the field. Once a class in defined, we can define (_instatiate_) objects of this class, that will act as "superpowered" variables in your program.

Take a moment to write down what attributes and methods are neededed in *spatial_volume*. Always try to be as simple as possible.

Probably you have proposed something similar to:

_Atributes_
- xlim, ylim, zlim: the limits of the box will extend over the intervals xlim, ylim, zlim. Each interval is a tuple: xlim=(x_min, x_max), etc.
- nx, ny, nz: number of points in which each of the dimensions is discretized.
- dx, dy, dz: spatial elements along each of the coordinates.

_Methods_
- get_x_axis: provides for a vector extending from (-xlim, xlim) with nx elements.
- get_y_axis: provides for a vector extending from (-ylim, ylim) with ny elements.
- get_z_axis: provides for a vector extending from (-zlim, zlim) with nz elements.
- get_xy_mesh: provides for a meshgrid in the xy plane
- get_zy_mesh: provides for a meshgrid in the zy plane
- get_zx_mesh: provides for a meshgrid in the zx plane

Each class must inlude a method to be called when initializing an object of the class. This method is always called \_\_init\_\_ and will contain the definition of all the attributes.

Follow this simple scheme to define your *spatial_volume* class:

```python
class class_name:
    def __init__(self, value1, value2, value3):
        self.attribute1= value1
        self.attribute2= value1
        self.attribute3= value3
````

Once you have your proposal, execute the cell (SHIFT-RETURN or Enter) to be sure your code is syntactically correct

> <font color='red'>Spoiler alert! Probably you will have come out with something like

```python
class SpatialVolume:
    def __init__(self, xlim, ylim, zlim, nx, ny, nz):
        self.xlim = xlim
        self.ylim = ylim
        self.zlim = zlim
        self.nx = nx
        self.ny = ny
        self.nz = nz

        self.dx=(xlim[1]-xlim[0])/nx  # xlim=(x_min, x_max) is a tuple
        self.dy=(ylim[1]-ylim[0])/ny
        self.dz=(zlim[1]-zlim[0])/nz
````
Once defined, we can define objects of this class. For each object we can access to the attributes:

```python
class class_name:
    def __init__(self, value1, value2, value3):
        self.attribute1= value1
        self.attribute2= value1
        self.attribute3= value3

object_V=class_name(v1,v2,v3)
object_W=class_name(w1,w2,w3)

print(object_V.attribute1)  # v1
print(object_W.attribute2)  # v2
````

Define two SpatialVolume objects and access to the attributes in the following cell

Now let us define in SpatialVolume some methods. A method is a function associated with the object. It is defined as the following

```python
class class_name:
    def __init__(self, value1, value2, value3):
        self.attribute1= value1
        self.attribute2= value1
        self.attribute3= value3

    def method1(self, param1, param2):
        result=self.attribute1*param1+self.attribute2/param2
        return result

````

Let us know incorporate to the object _SpatialVolume_ the method _get_x_axis_, that provides for a vector extending form (-xlim, xlim) with nx elements. The function, therefore, will return
```python
np.linspace(x_min, x_max , nx)
````

Extend the _SpatialVolume_ object  to incorporate the methods *get_x_axis*, *get_y_axis* and *get_z_axis*

> <font color='red'>Spoiler alert! Probably you will have come out with something like

```python
class SpatialVolume:
    def __init__(self, xlim, ylim, zlim, nx, ny, nz):
        self.xlim = xlim
        self.ylim = ylim
        self.zlim = zlim
        self.nx = nx
        self.ny = ny
        self.nz = nz

        self.dx=(xlim[1]-xlim[0])/nx  # xlim=(x_min, x_max) is a tuple
        self.dy=(ylim[1]-ylim[0])/ny
        self.dz=(zlim[1]-zlim[0])/nz
    
    def get_x_axis(self):
        return np.linspace(self.xlim[0], self.xlim[1], self.nx)
    
    def get_y_axis(self):
        return np.linspace(self.ylim[0], self.ylim[1], self.ny)

    def get_z_axis(self):
        return np.linspace(self.zlim[0], self.zlim[1], self.nz)
````

This already provides for the coordinates in each axis. You can use this data, for instance, to generate a 1D plot of any funtion over the volume dimension. Copy the following to a code cell and execute it (be sure you have already executed the definition of the _SpatialVolume_ class)

```python

V=SpatialVolume((-1,1),(-2,2),(-3,3), 50, 50, 50) 
x=V.get_x_axis()
fx=x**2
y=V.get_y_axis()
fy=np.sin(2*y)

z=V.get_y_axis()
fz=np.sin(3*z)

fig,ax=plt.subplots(1,1, figsize=(5,2))
ax.plot(x,fx)
ax.plot(y,fy)
ax.plot(z,fz)

````

An extended funtionality for the SpatialVolume class is to incude methods that provide 2D meshgrids in the planes xy, zy, zx so we can use them for 2D plots. A mesh grid is the 2D counterpart of a 1D axis. It is composed by two 2D arrays, one associated with the first axis and the other to the second. To understand this better lets consider a typical 2D surface plot:

```python
x = np.linspace(-5, 5, 50)  #create x axis
y = np.linspace(-5, 5, 50) # create y axis
X, Y = np.meshgrid(x, y, indexing='xy') # create xy meshgrid

f= np.sin(np.sqrt(X**2 + Y**2))  # define the function over the X and Y meshgrid coordinates

# 3. Plot the surface
from mpl_toolkits.mplot3d import Axes3D  # Needed for 3D plots

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Surface plot
surf = ax.plot_surface(X, Y, f, cmap='viridis')

# Optional: Add a color bar
fig.colorbar(surf)

plt.title('Example Surface Plot using meshgrid')
plt.show()
````

Following these ideas, incorporate the methods get_xy_mesh, get_zx_mesh, get_zy_mesh.

> <font color='red'>Spoiler alert! Your final *SpatialVolume* class should be something similar to

```python
class SpatialVolume:
    def __init__(self, xlim, ylim, zlim, nx, ny, nz):
        self.xlim = xlim
        self.ylim = ylim
        self.zlim = zlim
        self.nx = nx
        self.ny = ny
        self.nz = nz

        self.dx=(xlim[1]-xlim[0])/nx  # xlim=(x_min, x_max) is a tuple
        self.dy=(ylim[1]-ylim[0])/ny
        self.dz=(zlim[1]-zlim[0])/nz
    
    def get_x_axis(self):
        return np.linspace(self.xlim[0], self.xlim[1], self.nx)
    
    def get_y_axis(self):
        return np.linspace(self.ylim[0], self.ylim[1], self.ny)

    def get_z_axis(self):
        return np.linspace(self.zlim[0], self.zlim[1], self.nz)
    
    def get_xy_mesh(self):
        x=self.get_x_axis()
        y=self.get_y_axis()
        return np.meshgrid(x,y, indexing='xy')
    
    def get_xy_mesh(self):
        x=self.get_x_axis()
        y=self.get_y_axis()
        return np.meshgrid(x,y, indexing='xy')
    
    def get_zy_mesh(self):
        z=self.get_z_axis()
        y=self.get_y_axis()
        return np.meshgrid(z,y, indexing='xy')


## The class _LaserBeam_

Now it is time to work with the electromagnetic field $U(\mathbf{r})$. For this we should create a class named *LaserBeam*. We shall use different types of laser beams, that are described with different paramenters. This has a dificulty in the definition of the *\_\_init\_\_* method, as the parameters for initialization will depend on the laser beam type. We shall solve this by passing the parameters as a whole, in the form of a Python _dictionary_.

A Python dictionary is a structure that associates a _value_ with a given tag or _key_. It works as in the following example

```python
fruits_price_per_kg={'orange': 5, 'apple': 4, 'banana': 4.5}

# dictionay structure
print(fruits_price_per_kg.items()) # dict_items([('orange', 5), ('apple', 4), ('banana', 4.5)])
print(fruits_price_per_kg.keys())  # dict_keys(['orange', 'apple', 'banana'])
print(fruits_price_per_kg.values())  # dict_values([5, 4, 4.5])

# get a value from a key
print(fruits_price_per_kg['banana']) # 4.5

# check is a key is defined in the dictionary
print('strawberry' in fruits_price_per_kg) # False
print('apple' in fruits_price_per_kg) #True

# incorporate a new key/value pair
fruits_price_per_kg['pineapple']=4
print(fruits_price_per_kg.items()) # dict_items([('orange', 5), ('apple', 4), ('banana', 4.5), ('pineapple', 4)])
````

Now, some of the parameters that are common in laser beams are: type, amplitude, waist, wavelength, phase0. Other parameters, however, will depend on the laser beam type.

So, let us start defining our _LaserBeam_ class initialization using a dictionary with the parameters, and also including the SpatialVolume where it is defined. Try to do it in the following cell

> <font color='red'>Spoiler alert! Probably you will have come out with something like


```python
class LaserBeam:
    def __init__(self, parameters, volume):
        self.volume=volume
        self.parameters=parameters
````

and example on object initialization will be 

```python
class SpatialVolume:
    def __init__(self, xlim, ylim, zlim, nx, ny, nz):
        self.xlim = xlim
        self.ylim = ylim
        self.zlim = zlim
        self.nx = nx
        self.ny = ny
        self.nz = nz

        self.dx=(xlim[1]-xlim[0])/nx  # xlim=(x_min, x_max) is a tuple
        self.dy=(ylim[1]-ylim[0])/ny
        self.dz=(zlim[1]-zlim[0])/nz
    
    def get_x_axis(self):
        return np.linspace(self.xlim[0], self.xlim[1], self.nx)
    
    def get_y_axis(self):
        return np.linspace(self.ylim[0], self.ylim[1], self.ny)

    def get_z_axis(self):
        return np.linspace(self.zlim[0], self.zlim[1], self.nz)
    
    def get_xy_mesh(self):
        x=self.get_x_axis()
        y=self.get_y_axis()
        return np.meshgrid(x,y, indexing='xy')
    
    def get_xy_mesh(self):
        x=self.get_x_axis()
        y=self.get_y_axis()
        return np.meshgrid(x,y, indexing='xy')
    
    def get_zy_mesh(self):
        z=self.get_z_axis()
        y=self.get_y_axis()
        return np.meshgrid(z,y, indexing='xy')

class LaserBeam:
    def __init__(self, parameters, volume):
        self.volume=volume
        self.parameters=parameters
        
parameters={
    'beam_type':'LaguerreGauss',
    'wavelength':633e-9,
    'amplitude':1,
    'waist':1e-3,
    'phase0':0,
    'l': 1, 
    'p': 0
    }

V=SpatialVolume(xlim=(-2e-3, 2e-3), ylim=(-2e-3, 2e-3), zlim=(-20, 20), nx=300, ny=300, nz=600)
    
laserbeam1=LaserBeam(parameters, V)

````




## Orbital angular momentum: Laguerre Gaussian Modes

#### The angular momentum of light

One can identify a cornerstone in the development of structured light when Allen and co-workers identified in 1992 the helical phase structure of a laser beam with the orbital angular momentum (OAM) of photons <a href="https://journals.aps.org/pra/abstract/10.1103/PhysRevA.45.8185">(Allen et al. 1992)</a>. This simple but revolutionary idea boosted the application of structured light beams to many different fields, such as quantum optics, cold atoms, optical communications, microscopy, imaging, metrology, or nonlinear optics, among others <a href="https://www.nature.com/articles/s41566-021-00780-4">(Forbes et al. 2021)</a>. 

Light beams exhibit two degrees of freedom associated to their angular momentum: spin angular momentum (SAM), indicating the direction in which the field oscillates in time, or polarization; and orbital angular momentum (OAM) related to the spatial profile of the phase of the electric wave. SAM is a microscopic property of light fields, characterized by the spin, $\sigma=\pm 1$ , and defining the light field polarization, from right ($\sigma= 1$) or left ($\sigma=-1$) circularly polarized, to elliptically or linearly polarized depending on the photon spin combination. On the other hand, OAM is related to the transversal spiral-phase structure of a light beam, representing a macroscopic property that that is characterized by the topological charge, $\ell$—the number of $2\pi$-phase shifts along the azimuthal coordinate. Remarkably, while the spin is restricted to $\pm1$, the topological charge can take infinite discrete values.

Light beams carrying OAM can be naturally described through the well known Laguerre-Gauss modes. LG modes are solutions of the Helmholtz equation in cylindrical coordinates, and in this notebook we employ them to describe a laser beam with a vortex spatial structure. \
We shall consider an electric field linearly polarized along the $x$-direction, propagating in the $z$-direction under the paraxial approximation, and with wavelength $\lambda_0\ (k_0=2\pi/\lambda_0)$, expressed as 

\begin{equation}
U({\bf r})= U_0 LG_{\ell,p}({\bf r}) e^{-i \phi_0} 
\end{equation}

The spatial dependence is given by the Laguerre-Gauss modes in cylindrical coordinates as

\begin{equation}
LG_{\ell,p}(\rho, \phi, z)={w_0 \over w(z)} \left( {\sqrt{2} \rho \over w(z) }\right)^{|\ell|} L_p^{|\ell|} \left( {2\rho^2 \over w^2(z)}\right) e^{ - \rho^2 \over w^2(z)} e^{i\varphi_s(\rho,\phi,z)},
\end{equation}

where $\varphi_s(\rho,\phi,z)=\ell\phi + {k_0\rho^2\over 2R(z)}+\Phi_G (z)$ is the spatial phase of each mode. The beam waist is given by $w(z)=w_0\sqrt{1+(z/z_R)^2}$ with $w_0$ the beam waist at focus and $z_R=k_0w_0^2/2$ the Rayleigh distance. $R(z)=z[1+(z_R/z)^2]$ is the radius of curvature, $\Phi_G(z)=-(2p+|\ell|+1)\arctan(z/z_R)$ the Gouy phase, and $L_p^{|\ell|}$ the associated Laguerre polynomials. The indices $\ell=0, \pm 1, \pm 2, ...$ and $p=0,1,2,...$ correspond to the topological charge and the number of non-axial radial nodes of the mode, respectively. 

Now, let us introduce the definition of a Laguerre Gauss beam as a _type_ option in the class _LaserBeam_. We would like to define a method that returns the array of the complex field amplitude in the xy plane at the position z0 of the propagation axis. For now, we will just implement the field type "LaguerreGauss". 

Your _LaserBeam_ class must have, therefore, the following structure 

```python
class LaserBeam:
    def __init__(self, parameters, volume):
        self.volume=volume
        self.parameters=parameters

    def FieldAmplitude(self, z0):
        beam_type=self.parameters['beam_type']
        wavelength=self.parameters['wavelength']
        amplitude=self.parameters['amplitude']
        waist=self.parameters['waist']
        phase0=self.parameters['phase0']
        k=2*np.pi/wavelength
        
        X, Y=self.volume.get_xy_mesh()
        
        if beam_type=='LaguerreGauss':
            .
            .
            .

        return X,Y, amp*exp_phase 

````

Try to implement the LaguerreGauss beam inside the conditional block. To test your results use this code

```python
V=SpatialVolume(xlim=(-2e-3, 2e-3), ylim=(-2e-3, 2e-3), zlim=(-20, 20), nx=300, ny=300, nz=600)
    
beam1=LaserBeam(parameters, V)

X,Y,U=beam1.FieldAmplitude(z0=0)


fig,ax=plt.subplots(1,1, figsize=(4,4))

ax.pcolormesh(X, Y, abs(U), shading='auto')
```` 

> <font color='red'>Spoiler alert! If you have been succesful, probably you will have ended with the following code (now including the whole script)

```python

import numpy as np 
import matplotlib.pyplot as plt
from scipy.special import hermite
from scipy.special import genlaguerre

class SpatialVolume:
    def __init__(self, xlim, ylim, zlim, nx, ny, nz):
        self.xlim = xlim
        self.ylim = ylim
        self.zlim = zlim
        self.nx = nx
        self.ny = ny
        self.nz = nz

        self.dx=(xlim[1]-xlim[0])/nx  # xlim=(x_min, x_max) is a tuple
        self.dy=(ylim[1]-ylim[0])/ny
        self.dz=(zlim[1]-zlim[0])/nz
    
    def get_x_axis(self):
        return np.linspace(self.xlim[0], self.xlim[1], self.nx)
    
    def get_y_axis(self):
        return np.linspace(self.ylim[0], self.ylim[1], self.ny)

    def get_z_axis(self):
        return np.linspace(self.zlim[0], self.zlim[1], self.nz)
    
    def get_xy_mesh(self):
        x=self.get_x_axis()
        y=self.get_y_axis()
        return np.meshgrid(x,y, indexing='xy')
    
    def get_xy_mesh(self):
        x=self.get_x_axis()
        y=self.get_y_axis()
        return np.meshgrid(x,y, indexing='xy')
    
    def get_zy_mesh(self):
        z=self.get_z_axis()
        y=self.get_y_axis()
        return np.meshgrid(z,y, indexing='xy')

class LaserBeam:
    def __init__(self, parameters, volume):
        self.volume=volume
        self.parameters=parameters

    def FieldAmplitude(self, z0):
        beam_type=self.parameters['beam_type']
        wavelength=self.parameters['wavelength']
        amplitude=self.parameters['amplitude']
        waist=self.parameters['waist']
        phase0=self.parameters['phase0']
        k=2*np.pi/wavelength
        
        X, Y=self.volume.get_xy_mesh()
        
        if beam_type=='LaguerreGauss':
            l=self.parameters['l']
            p=self.parameters['p']

            RHO=np.sqrt(X**2+Y**2)
            PHI= np.arctan2(Y, X)
            z_R = np.pi * waist**2 / wavelength  # Rayleigh range
            w_z = waist * np.sqrt(1 + (z0 / z_R)**2)  # beam radius

            if z0==0:
                R_z=np.inf
            else:
                R_z = z0 * (1 + (z_R / z0)**2)
        
            phi_G = - (2*p + abs(l) + 1) * np.arctan(z0 / z_R)  # Gouy phase

            RHO_n = np.sqrt(2) * RHO / w_z
            laguerre = genlaguerre(p, abs(l))(RHO_n**2)

            amp = amplitude*(waist / w_z) * np.exp(-RHO**2 / w_z**2) * (RHO_n**abs(l)) * laguerre
            exp_phase = np.exp(1j * (k * RHO**2 / (2 * R_z) + l * PHI + phi_G + phase0))

        return X,Y, amp*exp_phase 


parameters={
    'beam_type':'LaguerreGauss',
    'wavelength':633e-9,
    'amplitude':1,
    'waist':1e-3,
    'phase0':0,
    'l': 1, 
    'p': 0
    }

V=SpatialVolume(xlim=(-2e-3, 2e-3), ylim=(-2e-3, 2e-3), zlim=(-20, 20), nx=300, ny=300, nz=600)
    
beam1=LaserBeam(parameters, V)

X,Y,U=beam1.FieldAmplitude(z0=0)


fig,ax=plt.subplots(1,1, figsize=(4,4))

ax.pcolormesh(X, Y, abs(U), shading='auto')

````

> <font color='magenta'>In the previous plot we've represented the absolute value of the electric field. Try to visualize the phase, replacing "abs(U)" by "np.angle(U)".

## Visualization

Next step is to implement a method to visualize the electromagnetic field. For this we will use color density plots. Try it! and add the following method in _LaserBeam_ class

```python
    def density_plot(self, z0):
        
        X,Y,U=self.FieldAmplitude(z0)
        
        fig, ax = plt.subplots(1,2,figsize=(10, 4))
        
        ax[0].set_aspect((X.max() - X.min()) / (Y.max() - Y.min()))

        c = ax[0].pcolormesh(X, Y, abs(U)**2, shading='auto', cmap='viridis') 
        ax[0].set_xlabel(f'x [a.u.]')
        ax[0].set_ylabel(f'y [a.u.]')
        ax[0].set_title(f'$|U|^2$ in xy Plane at z0={z0:.4f}', pad=10)

        fig.colorbar(c, ax=ax[0])

        c = ax[1].pcolormesh(X, Y, np.angle(U), shading='auto', cmap='hsv',
                          vmin=-np.pi, vmax=np.pi)

        fig.colorbar(c, ax=ax[1])

        ax[1].set_xlabel(f'x [a.u.]')
        ax[1].set_ylabel(f'y [a.u.]')
        ax[1].set_title(f'$\\varphi$ in xy Plane at z0={z0:.4f}', pad=10)

        plt.tight_layout()
        plt.show()
```

Check how this works

```python
V=SpatialVolume(xlim=(-2e-3, 2e-3), ylim=(-2e-3, 2e-3), zlim=(-20, 20), nx=300, ny=300, nz=600)
    
beam1=LaserBeam(parameters, V)

beam1.density_plot(z0=0)
```


## Advanced Exercise #1: Spatial phase evolution of the Laguerre-Gauss beams along the propagation direction

Up to now we have represented the Laguerre-Gauss mode at the focal plane (z0=0). However, the LG modes evolution along the propagation axis is very intetersting, as both the intensity and phase profiles evolve according to the equation we presented.
> <font color='magenta'>In order to understand the significance of the different phase terms of a Laguerre-Gauss beam, represent the transverse intensity and phase profiles of a LG mode at different propagation distances (z0). Try to represent this profiles within two times the Rayleigh distance, ie, from $-2z_R$ to $2z_R$.</font>

## Advanced Exercise #2: Necklace beam

The availability of Laguerre-Gauss or Hermite-Gauss modes open the possibility of much more complex and beautiful structured. For example, in the next figure we present phased necklaces beams, consisting on an intensity necklace profile, where each of the lobes present a different phase.

<img src="images/2Necklace.png" width=600 />


> <font color='magenta'>Your turn! Try to generate a necklace beam. (Hint: consider the superposition of two Laguerre-Gaussian beams). Once generated, create, and name your own structured laser beam! Creativity is in the air!

# Advanced Exercise #3: Vector beam

Up to now, in this notebook we have analyzed the spatial intensity and phase structure of Laguerre-Gauss modes, which can be characterized through their orbital angular momentum (OAM). But what about their spin angular momentum (SAM) related to their polarization state? We can define laser beams whose polarization state varies in their transverse profile: the so-called vector beams. A good review of vector beams can be found in <a href="https://opg.optica.org/aop/fulltext.cfm?uri=aop-1-1-1&id=176226">Zhan et al.</a>. Light beams with radial and azimuthal polarization distributions are the paradigm of vector beams. On one hand, radial vector beams are especially interesting due to the non-vanishing longitudinal electric field component present in tightly focusing systems, which allows one to sharply focus light below the diffraction limit This property has been greatly significant in fields suchvas laser machining, optimal plasmonic focusing, particle acceleration, and molecular orientation determination.  On the other hand, azimuthal vector beams can induce longitudinal magnetic fields with potential applications in spectroscopy and microscopy. In the next figure the intensity (backgorund color) and polarization state (small lines) of an azimuthally polarized and a radially polarized laser beam is depicted.

<img src="images/3VectorBeam.png" width=600 />

Though vector beams can present any polarization state in their profiles, let us know concentrate in linearly polarized vector beams (LPVB), that present an inhomogeneous polarization direction in their profile. There are two (or more) approaches to represent a LPVB:

1. A LPVB can be obtained from the superposition of two counter-rotating Laguerre Gauss beams. Considering the left and right circularly polarized unit vectors as $\textbf{e}_{LCP}=\textbf{e}_x+i\textbf{e}_y$ and $\textbf{e}_{RCP}=\textbf{e}_x-i\textbf{e}_y$, and taking into account the relationship
\begin{eqnarray}
 $e^{-i\phi}[\cos(\phi+\varphi_0)\textbf{e}_x+\sin(\phi+\varphi_0) \textbf{e}_y=e^{-i\phi}\textbf{e}_{LCP}+e^{-i\phi}e^{i2\phi_0}\textbf{e}_{RCP}]e^{-i\phi_0}/2$, 
\end{eqnarray}
a LPVB can be interpreted as a superposition of two vortex beams and counter-rotating circular polarization (LCP and RCP), carrying opposite topological charges.

2. Alternatively, the spatially varying polarization character of a LPVB can be described in cartesian coordinates by

\begin{eqnarray}
	\textbf{E}(\rho, \phi, z) &=& E(\rho, \phi, z) \cos(\phi+\varphi_0) \textbf{e}_x + E(\rho, \phi, z) \sin(\phi+\varphi_0) \textbf{e}_y
\end{eqnarray}

where $\phi$ is the azimuthal coordinate, and $\varphi_0$ is a parameter that determines the overall orientation of the polarization distribution. In particular, radial and azimuthal polarization distributions correspond to $\varphi_0=0$ and $\varphi_0=\pi/2$, respectively. A good approach for $E(\rho, \phi, z)$ consists on considering a Laguerre-Gauss $LG_{01}$ mode without the vortex phase term $e^{i\phi}$.


> <font color='magenta'>Your turn! Build your own vector beam. In particular, build radially and azimutally polarized LPVB. Think on how to represent their inhomogeneous polarization state! 

## Advanced Exercise #4: Optical skyrmions

Have you ever hear about an skyrmion? Maybe related to magnetism... but nowadays we can generate optical skyrmions too! An optical skyrmion is a structured light field whose polarization texture forms a topologically protected pattern that covers the entire Poincaré sphere of polarization states within a localized region of space. In other words, an optical skyrmion is a two- or three-dimensional configuration of the light’s polarization vector that maps every possible polarization state (linear, elliptical, circular) in a continuous, nontrivial way. This texture is robust against small perturbations because it is characterized by a topological number (the skyrmion number) that counts how many times the local polarization vectors wrap around the Poincaré sphere. In the next figure we present the intensity, polarization and phase distributions of an optical skyrmion.

<img src="images/4Skyrmion.png" width=600 />

> <font color='magenta'>Your turn! Within the code you've developed up to this point, try to generate an optical skyrmion. (No hints this time)